# Evalbids Order - A/B Test Analysis
### Workana Growth Team | Product Data Analyst Challenge

---

**Objetivo:** Evaluar si el nuevo orden de relevancia en Evalbids mejora la conversión del marketplace.

**Hipotesis:** Reordenar a los freelancers por un score de relevancia ponderado (gamificación 30%, proyectos en categoría 20%, etc.) facilitará al cliente la selección, impactando positivamente en conversión.

| Grupo | Descripcion |
|---|---|
| **Control** (`default`) | Orden de relevancia legacy |
| **Test** (`evalbidsNewOrder`) | Nuevo orden ponderado + filtros de calidad |

**Periodo:** 3 - 21 julio 2025 (18 días)

## 1. Setup y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

FILE = r'input/ChallengeDataAnalystEvalbidsOrderDatos.xlsx'

# Cargar todas las hojas
abtests       = pd.read_excel(FILE, sheet_name='abtests')
projects      = pd.read_excel(FILE, sheet_name='projects')
bids          = pd.read_excel(FILE, sheet_name='bids')
threads       = pd.read_excel(FILE, sheet_name='threads')
accepted_bids = pd.read_excel(FILE, sheet_name='accepted_bids')
payments      = pd.read_excel(FILE, sheet_name='payments')
skills        = pd.read_excel(FILE, sheet_name='skills')

print(f'Hojas cargadas: abtests({len(abtests)}), projects({len(projects)}), bids({len(bids)}), '
      f'threads({len(threads)}), accepted_bids({len(accepted_bids)}), payments({len(payments)}), skills({len(skills)})')

## 2. Construcción de la Tabla Maestra (nivel proyecto)

In [ ]:
# --- BASE: projects + grupo A/B ---
master = projects.merge(
    abtests[['project_id', 'segment']],
    left_on='id', right_on='project_id', how='inner'
).drop(columns=['project_id'])

# Etiqueta legible para el grupo
master['group'] = master['segment'].map({
    'default': 'Control',
    'evalbidsNewOrder': 'Test'
})

print(f'Proyectos en tabla maestra: {len(master)}')
print(f'  Control: {(master.group=="Control").sum()}')
print(f'  Test:    {(master.group=="Test").sum()}')

In [ ]:
# --- BIDS: cantidad de propuestas por proyecto ---
bids_per_project = bids.groupby('project_id').agg(
    n_bids=('id', 'count')
).reset_index()

master = master.merge(bids_per_project, left_on='id', right_on='project_id', how='left')
master.drop(columns=['project_id'], inplace=True, errors='ignore')
master['n_bids'] = master['n_bids'].fillna(0).astype(int)

# --- ACCEPTED BIDS: flag si tiene al menos un accepted bid ---
projects_with_ab = accepted_bids['project_id'].unique()
master['has_accepted_bid'] = master['id'].isin(projects_with_ab).astype(int)

# --- THREADS: total de mensajes por proyecto ---
messages_per_project = threads.groupby('project_id').agg(
    total_messages=('total_messages', 'sum'),
    n_threads=('id', 'count')
).reset_index()

master = master.merge(messages_per_project, left_on='id', right_on='project_id', how='left')
master.drop(columns=['project_id'], inplace=True, errors='ignore')
master['total_messages'] = master['total_messages'].fillna(0).astype(int)
master['n_threads'] = master['n_threads'].fillna(0).astype(int)

# --- PAYMENTS (via accepted_bids): flag pago completado + GMV ---
# Vincular payments con proyectos a través de accepted_bids
ab_pay = accepted_bids[['id', 'project_id']].merge(
    payments[payments['status'] == 'paid'][['accepted_bid_id', 'gross_gmv']],
    left_on='id', right_on='accepted_bid_id', how='inner'
)

gmv_per_project = ab_pay.groupby('project_id').agg(
    paid_gmv=('gross_gmv', 'sum'),
    n_paid_payments=('accepted_bid_id', 'count')
).reset_index()

master = master.merge(gmv_per_project, left_on='id', right_on='project_id', how='left')
master.drop(columns=['project_id'], inplace=True, errors='ignore')
master['paid_gmv'] = master['paid_gmv'].fillna(0)
master['n_paid_payments'] = master['n_paid_payments'].fillna(0).astype(int)
master['has_paid'] = (master['n_paid_payments'] > 0).astype(int)

# --- Estado productivo ---
productive_statuses = ['working', 'escrowing', 'finished', 'rating']
master['is_productive'] = master['status'].isin(productive_statuses).astype(int)

print('Columnas de la tabla maestra:')
print([c for c in master.columns])
print(f'\nShape: {master.shape}')

In [ ]:
# Vista rápida de la tabla maestra
display_cols = ['id', 'group', 'client_type', 'user_country', 'status', 'el1',
                'n_bids', 'has_accepted_bid', 'total_messages', 'has_paid', 'paid_gmv', 'is_productive']
master[display_cols].head(10)

## 3. Sanity Check - Integridad de la Tabla Maestra

In [ ]:
print('=== SANITY CHECKS ===')
print(f'\n1. Proyectos unicos: {master.id.nunique()} (filas: {len(master)})')
print(f'   -> Sin duplicados: {master.id.nunique() == len(master)}')

print(f'\n2. Distribución por grupo:')
print(master['group'].value_counts().to_string())

print(f'\n3. Coherencia: todos los has_paid=1 tienen has_accepted_bid=1?')
print(f'   -> {(master[master.has_paid==1].has_accepted_bid == 1).all()}')

print(f'\n4. Rango n_bids: {master.n_bids.min()} - {master.n_bids.max()}, mediana: {master.n_bids.median()}')
print(f'   Proyectos con 0 bids: {(master.n_bids == 0).sum()}')

print(f'\n5. Proyectos con accepted_bid: {master.has_accepted_bid.sum()} ({master.has_accepted_bid.mean()*100:.1f}%)')
print(f'   Proyectos con pago: {master.has_paid.sum()} ({master.has_paid.mean()*100:.1f}%)')
print(f'   Proyectos productivos: {master.is_productive.sum()} ({master.is_productive.mean()*100:.1f}%)')

## 4. Definición de Métricas

### Métrica de Éxito Principal: **Tasa de Accepted Bid**

$$\text{Accepted Bid Rate} = \frac{\text{Proyectos con al menos 1 accepted bid}}{\text{Total de proyectos}}$$

**Justificación:**

El experimento cambia el **orden en que se muestran los freelancers** en la página Evalbids. El impacto directo y más inmediato de este cambio es sobre la **decisión del cliente de aceptar una propuesta**. Esta métrica:

1. **Es la acción de conversión más cercana a la intervención**: El nuevo orden busca facilitar que el cliente encuentre y seleccione al freelancer adecuado. Aceptar un bid es la materialización de esa decisión.
2. **Tiene suficiente volumen para detectar diferencias**: Con 2,234 proyectos y ~24% de accepted bids, tenemos suficiente base para medir cambios significativos.
3. **No depende de factores externos al experimento**: A diferencia de pagos (que dependen de métodos de pago, presupuesto, etc.) o mensajes (que pueden reflejar confusión, no engagement positivo), el accepted bid es una decisión directa del cliente sobre la propuesta.
4. **Se alinea con el objetivo de negocio**: El PDF dice explícitamente "impactando directamente en conversión". En el funnel de Workana, aceptar un bid ES el momento de conversión clave.

---

### Métricas Complementarias

| Métrica | Fórmula | Qué nos dice |
|---|---|---|
| **Tasa EL1** | `sum(el1) / count(projects)` | Si el nuevo orden genera más engagement inicial. Un cliente que responde mensajes esta evaluando activamente a los freelancers. Es el paso previo a la conversión. |
| **Tasa de Pago (Paid Rate)** | `sum(has_paid) / count(projects)` | Conversión monetaria real. Valida que la aceptación se traduzca en revenue. Un accepted bid sin pago es conversión incompleta. |
| **Tasa Productiva** | `sum(is_productive) / count(projects)` | Proporción de proyectos que avanzan a estados de trabajo real. Complementa paid rate al incluir proyectos en proceso que aún no pagaron. |
| **GMV Promedio (pagados)** | `sum(paid_gmv) / count(has_paid=1)` | Ticket promedio de los proyectos que convierten. Si el nuevo orden favorece freelancers más caros/baratos, se reflejaría aquí. |
| **Mensajes Promedio por Proyecto** | `sum(total_messages) / count(projects)` | Nivel de interacción. Más mensajes puede indicar mejor engagement O más fricción. Hay que interpretar en contexto con las otras métricas. |
| **Bids Promedio por Proyecto** | `sum(n_bids) / count(projects)` | Métrica de control/guardrail. No debería cambiar entre grupos (los bids se envian antes de que el cliente vea el orden). Si cambia, algo está mal en la asignación. |

## 5. Resultados: Test vs Control

In [ ]:
def compute_metrics(df):
    """Computa todas las métricas para un grupo."""
    n = len(df)
    paid_projects = df[df['has_paid'] == 1]
    
    return pd.Series({
        'N proyectos': n,
        'Accepted Bid Rate (%)': df['has_accepted_bid'].mean() * 100,
        'Tasa EL1 (%)': df['el1'].mean() * 100,
        'Paid Rate (%)': df['has_paid'].mean() * 100,
        'Tasa Productiva (%)': df['is_productive'].mean() * 100,
        'GMV Promedio (USD, pagados)': paid_projects['paid_gmv'].mean() if len(paid_projects) > 0 else 0,
        'Mensajes Prom. por Proyecto': df['total_messages'].mean(),
        'Bids Prom. por Proyecto': df['n_bids'].mean(),
    })

# Calcular métricas por grupo
metrics = master.groupby('group').apply(compute_metrics, include_groups=False).T

# Agregar columna de diferencia relativa
metrics['Delta (%)'] = ((metrics['Test'] - metrics['Control']) / metrics['Control'] * 100)

# Formatear
print('\n' + '='*75)
print('        RESULTADOS DEL EXPERIMENTO: TEST vs CONTROL')
print('='*75)
print()

# Display con formato
styled = metrics.copy()
for col in ['Control', 'Test']:
    styled[col] = styled[col].apply(lambda x: f'{x:,.2f}')
styled['Delta (%)'] = metrics['Delta (%)'].apply(lambda x: f'{x:+.2f}%')

print(styled.to_string())

In [ ]:
# Visualizacion de las métricas de tasa (rates)
rate_metrics = ['Accepted Bid Rate (%)', 'Tasa EL1 (%)', 'Paid Rate (%)', 'Tasa Productiva (%)']

fig, axes = plt.subplots(1, 4, figsize=(16, 5))
colors = {'Control': '#5B8DB8', 'Test': '#E8834A'}

for i, metric in enumerate(rate_metrics):
    ax = axes[i]
    vals = [metrics.loc[metric, 'Control'], metrics.loc[metric, 'Test']]
    bars = ax.bar(['Control', 'Test'], vals, color=[colors['Control'], colors['Test']], 
                  width=0.5, edgecolor='white', linewidth=1.5)
    
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.3,
                f'{val:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    delta = metrics.loc[metric, 'Delta (%)']
    ax.set_title(metric.replace(' (%)', ''), fontsize=11, fontweight='bold')
    ax.set_ylabel('')
    ax.set_ylim(0, max(vals) * 1.25)
    
    # Delta annotation
    delta_color = '#2E7D32' if delta > 0 else '#C62828'
    ax.text(0.5, 0.95, f'Delta: {delta:+.1f}%', transform=ax.transAxes,
            ha='center', va='top', fontsize=9, color=delta_color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Métricas de Conversión: Test vs Control', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/01_rates_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Métricas de volumen/intensidad
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

volume_metrics = [
    ('Bids Prom. por Proyecto', 'Bids Promedio', ''),
    ('Mensajes Prom. por Proyecto', 'Mensajes Promedio', ''),
    ('GMV Promedio (USD, pagados)', 'GMV Promedio (USD)', '$')
]

for i, (metric, title, prefix) in enumerate(volume_metrics):
    ax = axes[i]
    vals = [metrics.loc[metric, 'Control'], metrics.loc[metric, 'Test']]
    bars = ax.bar(['Control', 'Test'], vals, color=[colors['Control'], colors['Test']],
                  width=0.5, edgecolor='white', linewidth=1.5)
    
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + max(vals)*0.02,
                f'{prefix}{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    delta = metrics.loc[metric, 'Delta (%)']
    delta_color = '#2E7D32' if delta > 0 else '#C62828'
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_ylim(0, max(vals) * 1.25)
    ax.text(0.5, 0.95, f'Delta: {delta:+.1f}%', transform=ax.transAxes,
            ha='center', va='top', fontsize=9, color=delta_color,
            bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Métricas de Volumen: Test vs Control', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/02_volume_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Análisis Estadístico Riguroso

A continuación evaluamos cada métrica con tests de significancia:
- **Tasas (proporciónes):** z-test de dos proporciónes
- **Medias continuas:** t-test de Welch (no asume varianzas iguales)

Nivel de significancia: **alpha = 0.05**

In [ ]:
from scipy import stats

# --- Colores del challenge ---
C_TEST = '#6C5CE7'
C_CTRL = '#A0A0A0'
ALPHA = 0.05

ctrl = master[master['group'] == 'Control']
test = master[master['group'] == 'Test']
n_c, n_t = len(ctrl), len(test)

def z_test_proportions(x_c, n_c, x_t, n_t):
    """Z-test para dos proporciónes independientes."""
    p_c = x_c / n_c
    p_t = x_t / n_t
    p_pool = (x_c + x_t) / (n_c + n_t)
    se = np.sqrt(p_pool * (1 - p_pool) * (1/n_c + 1/n_t))
    z = (p_t - p_c) / se if se > 0 else 0
    p_value = 2 * (1 - stats.norm.cdf(abs(z)))
    # CI de la diferencia (no pooled)
    se_diff = np.sqrt(p_c*(1-p_c)/n_c + p_t*(1-p_t)/n_t)
    diff = p_t - p_c
    ci_lo = diff - 1.96 * se_diff
    ci_hi = diff + 1.96 * se_diff
    return z, p_value, diff, ci_lo, ci_hi

def welch_t_test(arr_c, arr_t):
    """T-test de Welch para dos muestras independientes."""
    t_stat, p_value = stats.ttest_ind(arr_t, arr_c, equal_var=False)
    mean_diff = arr_t.mean() - arr_c.mean()
    se = np.sqrt(arr_t.var(ddof=1)/len(arr_t) + arr_c.var(ddof=1)/len(arr_c))
    ci_lo = mean_diff - 1.96 * se
    ci_hi = mean_diff + 1.96 * se
    return t_stat, p_value, mean_diff, ci_lo, ci_hi

# --- Calcular todos los tests ---
results = []

# 1. Accepted Bid Rate (z-test)
z, p, diff, ci_lo, ci_hi = z_test_proportions(
    ctrl['has_accepted_bid'].sum(), n_c, test['has_accepted_bid'].sum(), n_t)
results.append({
    'Métrica': 'Accepted Bid Rate *',
    'Tipo': 'Tasa',
    'Control': f"{ctrl['has_accepted_bid'].mean()*100:.2f}%",
    'Test': f"{test['has_accepted_bid'].mean()*100:.2f}%",
    'Diff (pp)': f"{diff*100:+.2f}",
    'Lift (%)': f"{(test['has_accepted_bid'].mean()/ctrl['has_accepted_bid'].mean()-1)*100:+.1f}%",
    'Stat': f"z={z:.3f}",
    'p-value': f"{p:.4f}",
    'IC 95% diff (pp)': f"[{ci_lo*100:+.2f}, {ci_hi*100:+.2f}]",
    'Sig?': 'Si' if p < ALPHA else 'No'
})

# 2. Tasa EL1 (z-test)
z, p, diff, ci_lo, ci_hi = z_test_proportions(
    ctrl['el1'].sum(), n_c, test['el1'].sum(), n_t)
results.append({
    'Métrica': 'Tasa EL1',
    'Tipo': 'Tasa',
    'Control': f"{ctrl['el1'].mean()*100:.2f}%",
    'Test': f"{test['el1'].mean()*100:.2f}%",
    'Diff (pp)': f"{diff*100:+.2f}",
    'Lift (%)': f"{(test['el1'].mean()/ctrl['el1'].mean()-1)*100:+.1f}%",
    'Stat': f"z={z:.3f}",
    'p-value': f"{p:.4f}",
    'IC 95% diff (pp)': f"[{ci_lo*100:+.2f}, {ci_hi*100:+.2f}]",
    'Sig?': 'Si' if p < ALPHA else 'No'
})

# 3. Paid Rate (z-test)
z, p, diff, ci_lo, ci_hi = z_test_proportions(
    ctrl['has_paid'].sum(), n_c, test['has_paid'].sum(), n_t)
results.append({
    'Métrica': 'Paid Rate',
    'Tipo': 'Tasa',
    'Control': f"{ctrl['has_paid'].mean()*100:.2f}%",
    'Test': f"{test['has_paid'].mean()*100:.2f}%",
    'Diff (pp)': f"{diff*100:+.2f}",
    'Lift (%)': f"{(test['has_paid'].mean()/ctrl['has_paid'].mean()-1)*100:+.1f}%",
    'Stat': f"z={z:.3f}",
    'p-value': f"{p:.4f}",
    'IC 95% diff (pp)': f"[{ci_lo*100:+.2f}, {ci_hi*100:+.2f}]",
    'Sig?': 'Si' if p < ALPHA else 'No'
})

# 4. Tasa Productiva (z-test)
z, p, diff, ci_lo, ci_hi = z_test_proportions(
    ctrl['is_productive'].sum(), n_c, test['is_productive'].sum(), n_t)
results.append({
    'Métrica': 'Tasa Productiva',
    'Tipo': 'Tasa',
    'Control': f"{ctrl['is_productive'].mean()*100:.2f}%",
    'Test': f"{test['is_productive'].mean()*100:.2f}%",
    'Diff (pp)': f"{diff*100:+.2f}",
    'Lift (%)': f"{(test['is_productive'].mean()/ctrl['is_productive'].mean()-1)*100:+.1f}%",
    'Stat': f"z={z:.3f}",
    'p-value': f"{p:.4f}",
    'IC 95% diff (pp)': f"[{ci_lo*100:+.2f}, {ci_hi*100:+.2f}]",
    'Sig?': 'Si' if p < ALPHA else 'No'
})

# 5. GMV Promedio pagados (Welch t-test, solo proyectos con pago)
ctrl_paid = ctrl[ctrl['has_paid']==1]['paid_gmv']
test_paid = test[test['has_paid']==1]['paid_gmv']
t_s, p, diff, ci_lo, ci_hi = welch_t_test(ctrl_paid, test_paid)
results.append({
    'Métrica': 'GMV Prom. (pagados)',
    'Tipo': 'Media',
    'Control': f"${ctrl_paid.mean():.2f}",
    'Test': f"${test_paid.mean():.2f}",
    'Diff (pp)': f"{diff:+.2f} USD",
    'Lift (%)': f"{(test_paid.mean()/ctrl_paid.mean()-1)*100:+.1f}%",
    'Stat': f"t={t_s:.3f}",
    'p-value': f"{p:.4f}",
    'IC 95% diff (pp)': f"[{ci_lo:+.2f}, {ci_hi:+.2f}] USD",
    'Sig?': 'Si' if p < ALPHA else 'No'
})

# 6. Mensajes promedio (Welch t-test)
t_s, p, diff, ci_lo, ci_hi = welch_t_test(ctrl['total_messages'], test['total_messages'])
results.append({
    'Métrica': 'Mensajes Prom.',
    'Tipo': 'Media',
    'Control': f"{ctrl['total_messages'].mean():.2f}",
    'Test': f"{test['total_messages'].mean():.2f}",
    'Diff (pp)': f"{diff:+.2f}",
    'Lift (%)': f"{(test['total_messages'].mean()/ctrl['total_messages'].mean()-1)*100:+.1f}%",
    'Stat': f"t={t_s:.3f}",
    'p-value': f"{p:.4f}",
    'IC 95% diff (pp)': f"[{ci_lo:+.2f}, {ci_hi:+.2f}]",
    'Sig?': 'Si' if p < ALPHA else 'No'
})

# 7. Bids promedio (guardrail, Welch t-test)
t_s, p, diff, ci_lo, ci_hi = welch_t_test(ctrl['n_bids'], test['n_bids'])
results.append({
    'Métrica': 'Bids Prom. (guardrail)',
    'Tipo': 'Media',
    'Control': f"{ctrl['n_bids'].mean():.2f}",
    'Test': f"{test['n_bids'].mean():.2f}",
    'Diff (pp)': f"{diff:+.2f}",
    'Lift (%)': f"{(test['n_bids'].mean()/ctrl['n_bids'].mean()-1)*100:+.1f}%",
    'Stat': f"t={t_s:.3f}",
    'p-value': f"{p:.4f}",
    'IC 95% diff (pp)': f"[{ci_lo:+.2f}, {ci_hi:+.2f}]",
    'Sig?': 'Si' if p < ALPHA else 'No'
})

results_df = pd.DataFrame(results)

print('='*110)
print('  TABLA DE RESULTADOS ESTADÍSTICOS  |  alpha = 0.05  |  * = Métrica principal')
print('='*110)
print(results_df.to_string(index=False))

## 8. Análisis de Funnel Completo

Proyecto creado -> EL1 (cliente responde) -> Accepted Bid -> Pago completado

Mostramos tasas absolutas (desde proyectos) Y tasas de paso entre etapas.

In [ ]:
# --- FUNNEL: Tasas absolutas y de paso ---
funnel_stages = ['Proyecto', 'EL1', 'Accepted Bid', 'Pago']

def build_funnel(df):
    n = len(df)
    el1 = df['el1'].sum()
    ab = df['has_accepted_bid'].sum()
    paid = df['has_paid'].sum()
    return [n, el1, ab, paid]

ctrl_funnel = build_funnel(ctrl)
test_funnel = build_funnel(test)

# Tabla de funnel
funnel_data = []
for i, stage in enumerate(funnel_stages):
    c_abs = ctrl_funnel[i] / ctrl_funnel[0] * 100
    t_abs = test_funnel[i] / test_funnel[0] * 100
    if i == 0:
        c_step, t_step = 100.0, 100.0
    else:
        c_step = ctrl_funnel[i] / ctrl_funnel[i-1] * 100 if ctrl_funnel[i-1] > 0 else 0
        t_step = test_funnel[i] / test_funnel[i-1] * 100 if test_funnel[i-1] > 0 else 0
    funnel_data.append({
        'Etapa': stage,
        'Control (n)': ctrl_funnel[i],
        'Control (% abs)': f"{c_abs:.1f}%",
        'Control (% paso)': f"{c_step:.1f}%",
        'Test (n)': test_funnel[i],
        'Test (% abs)': f"{t_abs:.1f}%",
        'Test (% paso)': f"{t_step:.1f}%",
    })

funnel_df = pd.DataFrame(funnel_data)
print("FUNNEL COMPLETO: Tasas absolutas y de paso entre etapas")
print("=" * 95)
print(funnel_df.to_string(index=False))

# --- Grafico de funnel ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Tasas absolutas
x = np.arange(len(funnel_stages))
w = 0.32
ctrl_abs = [v / ctrl_funnel[0] * 100 for v in ctrl_funnel]
test_abs = [v / test_funnel[0] * 100 for v in test_funnel]

bars_c = ax1.bar(x - w/2, ctrl_abs, w, label='Control', color=C_CTRL, edgecolor='white', linewidth=1.2)
bars_t = ax1.bar(x + w/2, test_abs, w, label='Test', color=C_TEST, edgecolor='white', linewidth=1.2)

for bars, vals in [(bars_c, ctrl_abs), (bars_t, test_abs)]:
    for bar, val in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(funnel_stages)
ax1.set_ylabel('% de Proyectos')
ax1.set_title('Tasas Absolutas por Etapa del Funnel', fontweight='bold')
ax1.legend()
ax1.set_ylim(0, 115)

# Tasas de paso (step rates) - excluyendo la primera etapa
step_stages = ['EL1\n(de Proyecto)', 'Accepted Bid\n(de EL1)', 'Pago\n(de Accepted)']
ctrl_steps = [ctrl_funnel[i]/ctrl_funnel[i-1]*100 for i in range(1, len(funnel_stages))]
test_steps = [test_funnel[i]/test_funnel[i-1]*100 for i in range(1, len(funnel_stages))]

x2 = np.arange(len(step_stages))
bars_c2 = ax2.bar(x2 - w/2, ctrl_steps, w, label='Control', color=C_CTRL, edgecolor='white', linewidth=1.2)
bars_t2 = ax2.bar(x2 + w/2, test_steps, w, label='Test', color=C_TEST, edgecolor='white', linewidth=1.2)

for bars, vals in [(bars_c2, ctrl_steps), (bars_t2, test_steps)]:
    for bar, val in zip(bars, vals):
        ax2.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 1,
                f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax2.set_xticks(x2)
ax2.set_xticklabels(step_stages)
ax2.set_ylabel('Tasa de Paso (%)')
ax2.set_title('Tasas de Conversión entre Etapas', fontweight='bold')
ax2.legend()
ax2.set_ylim(0, max(max(ctrl_steps), max(test_steps)) * 1.2)

plt.suptitle('Análisis de Funnel: Control vs Test', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/03_funnel_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Calidad del Freelancer Contratado (Gamificación)

In [ ]:
# Vincular accepted_bids con grupo A/B a través de project_id
ab_with_group = accepted_bids.merge(
    master[['id', 'group']], left_on='project_id', right_on='id', how='inner'
)

# Orden de gamificación
gam_order = ['iron', 'bronze', 'silver', 'gold', 'platinum', 'hero']
gam_labels = ['Iron', 'Bronze', 'Silver', 'Gold', 'Platinum', 'Hero']

# Distribución por grupo
gam_dist = pd.crosstab(
    ab_with_group['worker_position_gamification'],
    ab_with_group['group'],
    normalize='columns'
) * 100

# Reindexar por orden
gam_dist = gam_dist.reindex(gam_order)

print("Distribución de Gamificación de Freelancers Contratados (%):")
print("=" * 50)
for level in gam_order:
    c_val = gam_dist.loc[level, 'Control'] if level in gam_dist.index else 0
    t_val = gam_dist.loc[level, 'Test'] if level in gam_dist.index else 0
    print(f"  {level.capitalize():10s}  Control: {c_val:5.1f}%  |  Test: {t_val:5.1f}%")

# Proporción Gold+ (Gold, Platinum, Hero)
gold_plus = ['gold', 'platinum', 'hero']
ctrl_gp = ab_with_group[ab_with_group['group']=='Control']['worker_position_gamification'].isin(gold_plus).mean() * 100
test_gp = ab_with_group[ab_with_group['group']=='Test']['worker_position_gamification'].isin(gold_plus).mean() * 100
print(f"\n  Gold+ rate -> Control: {ctrl_gp:.1f}%  |  Test: {test_gp:.1f}%  |  Delta: {test_gp-ctrl_gp:+.1f}pp")

# --- Grafico ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Barras agrupadas
x = np.arange(len(gam_order))
w = 0.32
c_vals = [gam_dist.loc[g, 'Control'] if g in gam_dist.index else 0 for g in gam_order]
t_vals = [gam_dist.loc[g, 'Test'] if g in gam_dist.index else 0 for g in gam_order]

ax1.bar(x - w/2, c_vals, w, label='Control', color=C_CTRL, edgecolor='white', linewidth=1.2)
ax1.bar(x + w/2, t_vals, w, label='Test', color=C_TEST, edgecolor='white', linewidth=1.2)

for xi, (cv, tv) in enumerate(zip(c_vals, t_vals)):
    ax1.text(xi - w/2, cv + 0.5, f'{cv:.1f}%', ha='center', fontsize=8, color=C_CTRL, fontweight='bold')
    ax1.text(xi + w/2, tv + 0.5, f'{tv:.1f}%', ha='center', fontsize=8, color=C_TEST, fontweight='bold')

ax1.set_xticks(x)
ax1.set_xticklabels(gam_labels)
ax1.set_ylabel('% de Accepted Bids')
ax1.set_title('Distribución de Gamificación\n(Freelancers Contratados)', fontweight='bold')
ax1.legend()
ax1.axvline(x=2.5, color='red', linestyle='--', alpha=0.3, label='Umbral Gold+')

# Gold+ comparison
ax2.bar(['Control', 'Test'], [ctrl_gp, test_gp], color=[C_CTRL, C_TEST],
        width=0.5, edgecolor='white', linewidth=1.5)
ax2.text(0, ctrl_gp + 1, f'{ctrl_gp:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax2.text(1, test_gp + 1, f'{test_gp:.1f}%', ha='center', fontweight='bold', fontsize=13)
ax2.set_ylabel('% Gold+')
ax2.set_title('Tasa Gold+ en Freelancers Contratados\n(Gold + Platinum + Hero)', fontweight='bold')
ax2.set_ylim(0, max(ctrl_gp, test_gp) * 1.25)

delta_gp = test_gp - ctrl_gp
delta_color = '#2E7D32' if delta_gp > 0 else '#C62828'
ax2.text(0.5, 0.92, f'Delta: {delta_gp:+.1f}pp', transform=ax2.transAxes,
        ha='center', fontsize=11, color=delta_color, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

plt.suptitle('Calidad del Freelancer Contratado por Grupo', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/04_gamification_quality.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Segmentación por Client Type (New vs Rebuy)

In [ ]:
# --- Segmentación por client_type ---
seg_metrics = []
for ct in ['new', 'rebuy']:
    for grp in ['Control', 'Test']:
        sub = master[(master['client_type'] == ct) & (master['group'] == grp)]
        paid_sub = sub[sub['has_paid'] == 1]
        seg_metrics.append({
            'client_type': ct.capitalize(),
            'group': grp,
            'n': len(sub),
            'accepted_bid_rate': sub['has_accepted_bid'].mean() * 100,
            'el1_rate': sub['el1'].mean() * 100,
            'paid_rate': sub['has_paid'].mean() * 100,
            'productive_rate': sub['is_productive'].mean() * 100,
            'avg_messages': sub['total_messages'].mean(),
        })

seg_df = pd.DataFrame(seg_metrics)

# Tabla pivoteada para accepted_bid_rate
print("SEGMENTACION POR CLIENT TYPE")
print("=" * 80)
for ct in ['New', 'Rebuy']:
    subset = seg_df[seg_df['client_type'] == ct]
    c_row = subset[subset['group'] == 'Control'].iloc[0]
    t_row = subset[subset['group'] == 'Test'].iloc[0]
    lift = (t_row['accepted_bid_rate'] / c_row['accepted_bid_rate'] - 1) * 100

    # z-test para accepted bid rate
    c_sub = master[(master['client_type'] == ct.lower()) & (master['group'] == 'Control')]
    t_sub = master[(master['client_type'] == ct.lower()) & (master['group'] == 'Test')]
    z, p, diff, ci_lo, ci_hi = z_test_proportions(
        c_sub['has_accepted_bid'].sum(), len(c_sub),
        t_sub['has_accepted_bid'].sum(), len(t_sub)
    )

    print(f"\n  {ct} clients (Control n={int(c_row['n'])}, Test n={int(t_row['n'])})")
    print(f"    Accepted Bid Rate:  Control {c_row['accepted_bid_rate']:.1f}%  |  Test {t_row['accepted_bid_rate']:.1f}%  |  Lift {lift:+.1f}%  |  p={p:.4f} {'*' if p<0.05 else ''}")
    print(f"    EL1 Rate:           Control {c_row['el1_rate']:.1f}%  |  Test {t_row['el1_rate']:.1f}%")
    print(f"    Paid Rate:          Control {c_row['paid_rate']:.1f}%  |  Test {t_row['paid_rate']:.1f}%")
    print(f"    Productive Rate:    Control {c_row['productive_rate']:.1f}%  |  Test {t_row['productive_rate']:.1f}%")

# --- Grafico ---
fig, axes = plt.subplots(1, 4, figsize=(18, 6))
rate_cols = ['accepted_bid_rate', 'el1_rate', 'paid_rate', 'productive_rate']
rate_titles = ['Accepted Bid Rate', 'Tasa EL1', 'Paid Rate', 'Tasa Productiva']

for i, (col, title) in enumerate(zip(rate_cols, rate_titles)):
    ax = axes[i]
    x_pos = np.arange(2)
    w = 0.3

    for j, ct in enumerate(['New', 'Rebuy']):
        c_val = seg_df[(seg_df['client_type']==ct) & (seg_df['group']=='Control')][col].values[0]
        t_val = seg_df[(seg_df['client_type']==ct) & (seg_df['group']=='Test')][col].values[0]

        ax.bar(j - w/2, c_val, w, color=C_CTRL, edgecolor='white', linewidth=1.2,
               label='Control' if j==0 else '')
        ax.bar(j + w/2, t_val, w, color=C_TEST, edgecolor='white', linewidth=1.2,
               label='Test' if j==0 else '')

        ax.text(j - w/2, c_val + 0.5, f'{c_val:.1f}%', ha='center', fontsize=8, fontweight='bold')
        ax.text(j + w/2, t_val + 0.5, f'{t_val:.1f}%', ha='center', fontsize=8, fontweight='bold')

    ax.set_xticks(x_pos)
    ax.set_xticklabels(['New', 'Rebuy'])
    ax.set_title(title, fontweight='bold', fontsize=11)
    if i == 0:
        ax.legend(fontsize=9)

plt.suptitle('Métricas por Client Type: New vs Rebuy', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/05_segmentation_client_type.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Segmentación por País (Top 5)

In [ ]:
# Top 5 países por volumen total
top5_countries = master['user_country'].value_counts().head(5).index.tolist()

country_data = []
for country in top5_countries:
    for grp in ['Control', 'Test']:
        sub = master[(master['user_country'] == country) & (master['group'] == grp)]
        country_data.append({
            'country': country,
            'group': grp,
            'n': len(sub),
            'accepted_bid_rate': sub['has_accepted_bid'].mean() * 100 if len(sub) > 0 else 0,
            'paid_rate': sub['has_paid'].mean() * 100 if len(sub) > 0 else 0,
            'el1_rate': sub['el1'].mean() * 100 if len(sub) > 0 else 0,
        })

country_df = pd.DataFrame(country_data)

print("SEGMENTACION POR PAIS (Top 5) - Accepted Bid Rate")
print("=" * 85)
for country in top5_countries:
    c_row = country_df[(country_df['country']==country) & (country_df['group']=='Control')].iloc[0]
    t_row = country_df[(country_df['country']==country) & (country_df['group']=='Test')].iloc[0]
    lift = (t_row['accepted_bid_rate'] / c_row['accepted_bid_rate'] - 1) * 100 if c_row['accepted_bid_rate'] > 0 else float('inf')
    diff_pp = t_row['accepted_bid_rate'] - c_row['accepted_bid_rate']

    # z-test
    c_sub = master[(master['user_country']==country) & (master['group']=='Control')]
    t_sub = master[(master['user_country']==country) & (master['group']=='Test')]
    z, p, _, _, _ = z_test_proportions(
        c_sub['has_accepted_bid'].sum(), len(c_sub),
        t_sub['has_accepted_bid'].sum(), len(t_sub)
    )

    sig = '*' if p < 0.05 else ''
    print(f"  {country:4s}  Control: {c_row['accepted_bid_rate']:5.1f}% (n={int(c_row['n']):>4})  "
          f"|  Test: {t_row['accepted_bid_rate']:5.1f}% (n={int(t_row['n']):>4})  "
          f"|  Diff: {diff_pp:+.1f}pp  Lift: {lift:+.1f}%  p={p:.3f} {sig}")

# --- Grafico ---
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(top5_countries))
w = 0.32

c_rates = [country_df[(country_df['country']==c) & (country_df['group']=='Control')]['accepted_bid_rate'].values[0]
           for c in top5_countries]
t_rates = [country_df[(country_df['country']==c) & (country_df['group']=='Test')]['accepted_bid_rate'].values[0]
           for c in top5_countries]

bars_c = ax.bar(x - w/2, c_rates, w, label='Control', color=C_CTRL, edgecolor='white', linewidth=1.2)
bars_t = ax.bar(x + w/2, t_rates, w, label='Test', color=C_TEST, edgecolor='white', linewidth=1.2)

for xi, (cv, tv) in enumerate(zip(c_rates, t_rates)):
    ax.text(xi - w/2, cv + 0.3, f'{cv:.1f}%', ha='center', fontsize=9, fontweight='bold', color='#555')
    ax.text(xi + w/2, tv + 0.3, f'{tv:.1f}%', ha='center', fontsize=9, fontweight='bold', color=C_TEST)
    # Delta
    diff_pp = tv - cv
    color_d = '#2E7D32' if diff_pp > 0 else '#C62828'
    ax.text(xi, max(cv, tv) + 2.5, f'{diff_pp:+.1f}pp', ha='center', fontsize=8, color=color_d, fontweight='bold')

# N de cada pais como sublabel
n_labels = [f"n={master[master['user_country']==c].shape[0]}" for c in top5_countries]
ax.set_xticks(x)
ax.set_xticklabels([f"{c}\n({n})" for c, n in zip(top5_countries, n_labels)])
ax.set_ylabel('Accepted Bid Rate (%)')
ax.set_title('Accepted Bid Rate por País (Top 5)', fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.set_ylim(0, max(max(c_rates), max(t_rates)) * 1.35)

plt.tight_layout()
plt.savefig('output/06_segmentation_country.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Evolución Temporal Semanal

In [ ]:
# Crear columna de semana
master['created'] = pd.to_datetime(master['created'])
master['week'] = master['created'].dt.isocalendar().week.astype(int)
master['week_label'] = 'Sem ' + master['week'].astype(str)

# Calcular métricas semanales
weekly_data = []
for week in sorted(master['week'].unique()):
    for grp in ['Control', 'Test']:
        sub = master[(master['week'] == week) & (master['group'] == grp)]
        if len(sub) == 0:
            continue
        weekly_data.append({
            'week': week,
            'week_label': f'Sem {week}',
            'group': grp,
            'n': len(sub),
            'accepted_bid_rate': sub['has_accepted_bid'].mean() * 100,
            'el1_rate': sub['el1'].mean() * 100,
            'paid_rate': sub['has_paid'].mean() * 100,
        })

weekly_df = pd.DataFrame(weekly_data)

# Tabla
print("EVOLUCION SEMANAL - Accepted Bid Rate")
print("=" * 70)
for week in sorted(weekly_df['week'].unique()):
    c_row = weekly_df[(weekly_df['week']==week) & (weekly_df['group']=='Control')]
    t_row = weekly_df[(weekly_df['week']==week) & (weekly_df['group']=='Test')]
    if len(c_row) > 0 and len(t_row) > 0:
        c_row = c_row.iloc[0]
        t_row = t_row.iloc[0]
        diff = t_row['accepted_bid_rate'] - c_row['accepted_bid_rate']
        print(f"  Sem {week}  Control: {c_row['accepted_bid_rate']:5.1f}% (n={int(c_row['n']):>3})  "
              f"|  Test: {t_row['accepted_bid_rate']:5.1f}% (n={int(t_row['n']):>3})  "
              f"|  Diff: {diff:+.1f}pp")

# --- Grafico de lineas temporales ---
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
metrics_weekly = [
    ('accepted_bid_rate', 'Accepted Bid Rate (%)'),
    ('el1_rate', 'Tasa EL1 (%)'),
    ('paid_rate', 'Paid Rate (%)'),
]

for i, (col, title) in enumerate(metrics_weekly):
    ax = axes[i]
    for grp, color, marker in [('Control', C_CTRL, 'o'), ('Test', C_TEST, 's')]:
        grp_data = weekly_df[weekly_df['group'] == grp].sort_values('week')
        ax.plot(grp_data['week_label'], grp_data[col], marker=marker, linewidth=2.5,
                markersize=8, label=grp, color=color)
        # Etiquetas
        for _, row in grp_data.iterrows():
            offset = 1.2 if grp == 'Test' else -1.5
            ax.text(row['week_label'], row[col] + offset, f"{row[col]:.1f}%",
                   ha='center', fontsize=8, fontweight='bold', color=color)

    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.set_ylabel('%')
    ax.legend(fontsize=10)
    ax.tick_params(axis='x', rotation=0)

plt.suptitle('Evolución Temporal Semanal: Test vs Control', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/07_weekly_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. Conclusión y Toma de Decisiónes

---

### Qué podemos concluir de los resultados?

El nuevo orden de relevancia en Evalbids muestra una **señal positiva y consistente** en todo el funnel de conversión, pero **no alcanza significancia estadística** con el tamaño de muestra actual.

**La evidencia a favor:**

1. **Consistencia direcciónal total.** Las 4 métricas de conversión apuntan en la misma dirección: EL1 +3.9pp, Accepted Bid Rate +2.8pp, Paid Rate +3.1pp, Tasa Productiva +2.8pp. La probabilidad de que 4 métricas independientes se muevan a favor de Test por azar es baja (~6.25% si fueran independientes).

2. **El efecto se amplifica a lo largo del funnel.** El lift relativo crece desde EL1 (+6.2%) hasta Paid Rate (+16.6%). Esto no es ruido: indica que el nuevo orden no solo genera más engagement, sino que las propuestas que se aceptan en Test se concretan en pagos con mayor frecuencia (tasa de paso Accepted->Pago: 86.4% vs 83.3%).

3. **La calidad del freelancer contratado mejora.** La tasa Gold+ sube de 53.0% a 59.7% (+6.7pp). Bronze cae de 26.5% a 17.3%. El algoritmo está cumpliendo su función: poner mejores freelancers primero.

4. **EL1 y Paid Rate están en zona borderline** (p=0.056 y p=0.066 respectivamente). No alcanzamos alpha=0.05, pero estamos cerca.

5. **El guardrail está limpio.** Bids promedio no cambió (+2%, p=0.59), confirmando que la asignación al test no contaminó la oferta de freelancers.

**La evidencia en contra o de precaucion:**

1. **Ninguna métrica de conversión es significativa** al nivel convencional de 0.05. La más cercana es EL1 con p=0.056.

2. **La evolución semanal es irregular.** El efecto se concentra en Semana 28 (+6.5pp) y desaparece en Semana 29 (+0.1pp). No vemos un patrón estable semana a semana.

3. **México muestra efecto negativo** (-3.1pp), aunque con N muy bajo (n=130) y sin significancia (p=0.65).

4. **El GMV promedio por proyecto pagado baja -6%**, aunque no es significativo (p=0.66) y se compensa por el mayor volumen.

---

### Decisión: ITERAR - Extender el experimento antes de escalar

**No descartaría esta solución.** Los datos no permiten declarar un ganador estadístico, pero la consistencia direcciónal de las 4 métricas de conversión, la mejora en calidad de freelancers contratados, y los p-values borderline hacen que descartar seria un error. Hay demasiada señal como para atribuirlo a ruido.

**Tampoco escalaría a producción hoy.** Sin significancia estadística en la métrica principal, escalar conlleva riesgo: el efecto observado podría ser parcialmente aleatorio. El IC95% del Accepted Bid Rate va de -0.72pp a +6.37pp, lo que significa que el verdadero efecto podría ser nulo.

**Mi recomendación concreta: extender el test ~6 semanas mas** para alcanzar el poder estadístico necesario.

El power analysis lo confirma: con ~1,100 proyectos por grupo, el poder actual es **solo 35%**. Necesitamos ~3,600 por grupo para llegar a 80% de poder:

| Escenario | Proyectos/grupo | Poder | Tiempo adicional |
|---|---|---|---|
| **Actual (18 días)** | **~1,100** | **35%** | - |
| 80% de poder | ~3,586 | 80% | ~40 días (5.7 semanas) |
| 90% de poder | ~4,800 | 90% | ~59 días (8.5 semanas) |

*(A una tasa de ~62 proyectos/dia por grupo)*

**Justificación del framework de costos:**

- **Costo de extender = bajo:** mantener el 50/50 split unas semanas más no impacta significativamente el negocio.
- **Costo de escalar incorrectamente = alto:** si el efecto es ruido, degradamos la experiencia para todas las categorías.
- **Costo de descartar incorrectamente = el más alto:** si el efecto es real (+3pp en Accepted Bid Rate), estamos dejando ~60 conversiónes adicionales por cada 2,000 proyectos.

**Si al extender el test las métricas mantienen la dirección y alcanzan significancia**, escalaría con confianza a las 10 categorías del scope actual, priorizando clientes New donde el efecto es ~5x mayor.

## 14. Próximos Pasos

---

### Prioridad 1: Extender el experimento con ajustes de diseño

**Qué hacer:** Mantener el test activo ~6 semanas más (llegar a ~3,600 proyectos por grupo).

**Por qué:** El power analysis demuestra que con 1,100 proyectos/grupo solo teníamos 35% de poder. Con ~3,600 llegaremos a 80%, suficiente para detectar el efecto de ~3pp observado en Accepted Bid Rate. Los p-values borderline de EL1 (0.056) y Paid Rate (0.066) sugieren que estamos cerca de significancia: **no es que el efecto no exista, es que no tenemos suficiente muestra para confirmarlo.**

**Ajuste clave:** Definir el tamaño de muestra y la métrica de exito **antes** de extender (pre-registro). Esto evita el sesgo de "peeking": decidir cuándo parar en funcion de si los resultados son favorables.

| Parametro | Valor recomendado |
|---|---|
| Métrica principal | Accepted Bid Rate |
| Efecto mínimo detectable | 3pp (de 22.7% a 25.7%) |
| Alpha | 0.05 (two-sided) |
| Poder | 80% |
| N por grupo | 3,586 |
| Duración estimada | ~40 días adicionales |

---

### Prioridad 2: Analizar por categoría/subcategoría

**Qué hacer:** Desagregar los resultados por las 10 categorías del experimento (Web Design, Wordpress, SEO, E-commerce, etc.) para identificar dónde el nuevo orden funciona mejor y donde no.

**Por qué:** El análisis por pais reveló heterogeneidad (AR +52% vs MX -15%). Es muy probable que también exista heterogeneidad por categoría. El algoritmo pondera "proyectos trabajados en la categoría" (20%) y "proyectos ganados en la subcategoría" (20%), lo cual significa que su efectividad depende directamente de la densidad de freelancers con experiencia en cada vertical. Categorias como IT & Programming (alto volumen de freelancers especializados) probablemente se beneficien más que categorías de nicho como Engineering & Manufacturing - 3D Modelling.

**Impacto:** Si encontramos 3-4 categorías donde el efecto es fuerte y significativo, podríamos **escalar parcialmente** a esas categorías sin esperar resultados globales.

---

### Prioridad 3: Focalizar en clientes New como segmento principal

**Qué hacer:** Diseñar una segunda iteración donde el nuevo orden se aplique **solo a clientes New**, o con mayor agresividad para este segmento.

**Por qué:** El lift en clientes New es 5x mayor que en Rebuy (+17.7% vs +3.5%). Esto tiene lógica causal: un cliente nuevo no conoce la plataforma, no tiene freelancers favoritos, y depende mucho más del orden en que se le presentan las opciones. Un cliente rebuy ya sabe lo que busca. Este hallazgo sugiere que **el impacto del nuevo orden de relevancia está concentrado en quienes más lo necesitan**.

**Impacto:** Al focalizar en New, podríamos:
- Obtener significancia más rápido (efecto más grande = menor N necesario)
- Diseñar la experiencia de primer uso del marketplace de forma más efectiva
- Liberar a los rebuy del experimento, evitando posible fricción

---

### Prioridad 4: Investigar el aumento de Iron en accepted bids del grupo Test

**Qué hacer:** Analizar por que la proporción de freelancers Iron contratados sube de 9.8% a 12.4% en Test, a pesar de que el algoritmo debería priorizar freelancers de mayor nivel.

**Por qué:** El nuevo orden incluye filtros de calidad (Gold+, top 1000 ranking, etc.) para mostrar como "recomendados". Pero los clientes igualmente pueden scrollear y contratar a cualquiera. Si Iron sube, puede significar que:
- Algunos Iron ofrecen precios más competitivos y los clientes responden al precio
- El filtro de recomendados genera un efecto "contrast" donde los no-recomendados parecen más accesibles
- Hay proyectos donde no hay suficientes freelancers Gold+ y el algoritmo "rellena" con Iron

**Impacto:** Entender esto podría mejorar el algoritmo para evitar que clientes contraten freelancers de bajo nivel por default.

---

### Prioridad 5: Medir impacto en GMV total (no solo promedio)

**Qué hacer:** En la extensión del test, trackear **GMV total por grupo** (no solo promedio por proyecto pagado) como métrica complementaria.

**Por qué:** El GMV promedio baja -6% en Test ($137.8 vs $146.5), pero como la cantidad de pagos sube +16.6%, el **GMV total** del grupo Test es probablemente mayor. La pregunta de negocio real no es "cuánto paga cada proyecto" sino "cuánto genera el marketplace en total". Una baja de ticket compensada por mayor volumen de transacciónes es un trade-off positivo.

**Cálculo rápido:**
- Control: 215 pagos x $146.5 = ~$31,500 GMV total
- Test: 241 pagos x $137.8 = ~$33,200 GMV total
- **Test genera ~$1,700 más en GMV total (+5.4%)** a pesar del ticket promedio menor.

---

### Prioridad 6: Iterar los pesos del algoritmo

**Qué hacer:** Si el test se confirma positivo, experimentar con variaciones de los pesos relativos:
- Aumentar el peso de "skills matching" (actualmente solo 10%) dado que es la señal más directa de fit entre freelancer y proyecto
- Evaluar si Gamificación al 30% es demasiado peso para un proxy indirecto de calidad
- Testar versiones con pesos ajustados por categoría (ej: en categorías técnicas, dar más peso a proyectos en subcategoría)

**Por qué:** Los pesos actuales fueron definidos a priori sin datos. Ahora que tenemos evidencia de que el enfoque funciona, podemos optimizar los componentes. Un posible camino: usar los datos de pagos exitosos para hacer un análisis de regresión y derivar pesos óptimos basados en que predice mejor la conversión.

---

### Prioridad 7: Expandir el scope de categorías

**Qué hacer:** Una vez confirmado el efecto en las 10 categorías actuales, extender a todas las categorías de Workana de forma escalonada.

**Por qué:** El experimento actual cubre un subconjunto específico del marketplace. Si el efecto se confirma, hay una oportunidad de escalar el impacto a todo el trafico. La expansión escalonada permite monitorear el efecto en categorías muy diferentes (ej: Writing & Translation, Legal, Finance) donde la dinámica freelancer-cliente puede ser distinta.

---

### Resumen: Roadmap priorizado

| # | Acción | Horizonte | Impacto esperado |
|---|---|---|---|
| 1 | Extender el test 6 semanas | Inmediato | Confirmar/descartar efecto con rigor |
| 2 | Desagregar por categoría | En paralelo | Identificar dónde escalar primero |
| 3 | Focalizar en clientes New | Segunda iteración | Maximizar efecto donde es más fuerte |
| 4 | Investigar Iron en Test | En paralelo | Mejorar calidad del algoritmo |
| 5 | Medir GMV total | En extensión | Validar impacto en revenue |
| 6 | Iterar pesos del algoritmo | Post-confirmación | Optimizar la solución |
| 7 | Expandir a todas las categorías | Post-confirmación | Escalar impacto al marketplace |